# Images, Layers & the Dockerfile

## What's covered

- **What an image really is** — an ordered stack of read-only filesystem layers plus a JSON config blob
- **Layers and the union filesystem** — how `overlay2` glues layers into a single root, where the writable layer comes from
- **Image references** — `registry/namespace/repo:tag@digest`, what each piece defaults to
- **Inspecting images** — `docker image inspect`, `docker history`, reading a manifest
- **The Dockerfile** — every instruction worth knowing: `FROM`, `RUN`, `COPY`, `ADD`, `WORKDIR`, `ENV`, `ARG`, `EXPOSE`, `USER`, `LABEL`, `HEALTHCHECK`
- **`CMD` vs `ENTRYPOINT`** — the most-confused pair, made simple
- **The build context** — what `docker build .` actually sends to the daemon, and why `.dockerignore` matters
- **Multi-stage builds** — slim production images by throwing away the toolchain
- **BuildKit and `docker buildx`** — the modern build backend, what it changes
- **Tagging conventions** — `latest`, semver, digests, and how teams version images in practice

By the end you should be able to read any Dockerfile in the wild, write a production-quality one from scratch, and explain exactly what happens when you type `docker build`.

## What is an image, really?

In notebook 01 we said an image is "a frozen, read-only template — a snapshot of a filesystem plus a bit of metadata." That's true. Now let's be more precise.

A Docker image is two things bundled together:

1. **An ordered stack of filesystem layers.** Each layer is a tarball of file changes — additions, modifications, deletions — relative to the layer below it. The bottom layer might be "the entire Debian root filesystem"; the next might be "install Python"; the next might be "copy your app in."
2. **A JSON config blob** describing how to run the image — the default command (`CMD`), the entrypoint, the working directory, environment variables, exposed ports, the user to run as, labels, history.

These two pieces — the layers plus the config — are tied together by a **manifest**, which lists each layer's digest and the config's digest. The whole manifest is itself digested (SHA-256). That digest *is* the image's identity. When you say "`nginx:1.27.0`", a tag, you're really pointing at a digest like `sha256:abc123...`. The tag is a mutable nickname; the digest is the immutable truth.

This design — **content-addressable** layers — is what makes images cacheable. Two images that share a base layer share the *same bytes on disk*. Pull `python:3.12-slim` and then `python:3.12-slim-bookworm` and the overlap downloads once, not twice.

### What it isn't

A few common misconceptions worth clearing up:

- **An image is not a tarball of a complete OS.** It's just files. There's no init system inside; there's no kernel. When you "run" the image, your host kernel boots a process inside the union of those file layers.
- **An image is not an executable.** It's data. You need the Docker engine (or any OCI-compatible runtime) to interpret it.
- **An image doesn't "boot."** No kernel boot, no `init`, no systemd — just the host kernel exec-ing the configured command directly.

## Layers and the union filesystem

Every instruction in a Dockerfile (`FROM`, `RUN`, `COPY`, etc.) usually produces a new **layer** — a tarball of "what changed relative to the layer below me." Layers are stacked, bottom-up, and a **union filesystem driver** in the Docker daemon presents them to the container as a single root filesystem.

```
   +-----------------------------+
   |  writable container layer   |   <- changes a running container makes go here
   +-----------------------------+
   |  L4: COPY app.py /app/      |   read-only
   +-----------------------------+
   |  L3: pip install flask      |   read-only
   +-----------------------------+
   |  L2: apt install python3    |   read-only
   +-----------------------------+
   |  L1: debian:bookworm rootfs |   read-only
   +-----------------------------+
```

The default driver on Linux is `overlay2`, which uses the kernel's overlayfs feature to merge the directories with copy-on-write semantics:

- **Reads** fall through the stack — the file is served from the topmost layer that has it.
- **Writes** trigger a *copy-up* — the file is copied from its read-only layer into the writable layer, then modified there. The original stays untouched.
- **Deletes** are recorded as *whiteout files* in the writable layer that mask entries below.

Two big consequences:

1. **Layers are cached.** When you re-run `docker build` and Dockerfile instructions 1-7 haven't changed, the daemon reuses layers 1-7 from cache and only re-executes 8 onward. This is why Dockerfile order matters: put rarely-changing steps (installing dependencies) *before* frequently-changing steps (copying source code).
2. **The container's writable layer is ephemeral.** When you `docker rm` the container, that layer is deleted. Anything written to the container's filesystem — log files, uploaded user data, cached downloads — is gone. To persist data, you need a **volume** or a **bind mount**, which we cover in notebook 04.

## Image references — the full name of an image

When you wrote `docker run nginx`, you used a hugely abbreviated form. The full image reference is:

```
[registry-host[:port]/][namespace/]repository[:tag][@digest]
```

Each piece has a default. The four levels:

- **`nginx`** is equivalent to **`library/nginx`** (the `library/` namespace is where Docker Hub keeps official images).
- **`library/nginx`** is equivalent to **`docker.io/library/nginx`** (Docker Hub is the default registry).
- **`docker.io/library/nginx`** with no tag is equivalent to **`docker.io/library/nginx:latest`** (`latest` is the default tag — not "the newest", just "the tag named latest").
- And a fully-pinned reference looks like **`docker.io/library/nginx:1.27.0@sha256:abc123...`** — registry, namespace, repo, tag, *and* digest.

A few examples to read:

| Reference | Means |
|---|---|
| `ubuntu` | `docker.io/library/ubuntu:latest` |
| `python:3.12-slim` | `docker.io/library/python:3.12-slim` |
| `myorg/api:v2.1.0` | `docker.io/myorg/api:v2.1.0` |
| `ghcr.io/myorg/api:v2.1.0` | the same image hosted on GitHub Container Registry instead of Docker Hub |
| `registry.example.com:5000/internal/billing:prod` | private registry on port 5000 |
| `nginx@sha256:abc123...` | a specific immutable digest of `nginx`, regardless of which tag points there |

**Tag vs digest.** A tag like `1.27.0` *can* be re-pointed at a new image by the publisher (mutable). A digest is the SHA-256 of the manifest and is immutable by construction — pull `nginx@sha256:abc...` today and ten years from now, byte-for-byte the same image. Production deployments often pin by digest for this reason.

In [ ]:
# Pull a couple of images and look at what we have
!docker pull alpine:3.20
!docker pull python:3.12-slim
!echo '---'
!docker images | head -10

## Inspecting images — `inspect` and `history`

Two commands open the box and let you see what an image actually contains.

**`docker image inspect <ref>`** dumps the full JSON config: the layers (with digests), the default command, environment variables, exposed ports, user, working directory, labels, architecture, OS, and creation date. It's the source of truth for "what will this image do when I run it?"

**`docker image history <ref>`** shows the layers in order, with which Dockerfile instruction created each one and how big the layer is. This is gold for understanding *why* an image is the size it is.

The two together answer most "what's in this image?" questions without ever running it.

In [ ]:
# Look at the default command and exposed ports of nginx:alpine
!docker pull nginx:alpine
!docker image inspect nginx:alpine --format 'Cmd:        {{.Config.Cmd}}
Entrypoint: {{.Config.Entrypoint}}
WorkingDir: {{.Config.WorkingDir}}
User:       {{.Config.User}}
ExposedPorts: {{.Config.ExposedPorts}}
Env:        {{.Config.Env}}'
!echo '---'
# History: what layers built this image and how big each one is
!docker image history nginx:alpine --no-trunc | head -15

## Meet the Dockerfile

A **Dockerfile** is a plain text recipe for building an image. The file is conventionally named `Dockerfile` (no extension) and lives in the root of your project.

The simplest possible one:

```dockerfile
FROM alpine:3.20
CMD ["echo", "hello from the image"]
```

Save that as `Dockerfile`, then:

```bash
docker build -t my-hello .
docker run --rm my-hello
```

`docker build -t my-hello .` reads the `Dockerfile` in the current directory (`.`), executes its instructions in order, builds a new image, and tags it `my-hello:latest`. Then `docker run my-hello` runs it.

That's the whole picture. The rest of this notebook is "what instructions can go in a Dockerfile, and how do you choose them?"

### How a build executes (mental model)

Conceptually, each instruction either *adds a layer* or *modifies the config blob*:

- **Filesystem-changing instructions** (`RUN`, `COPY`, `ADD`) — execute in a temporary container based on the previous layer, then commit the result as a new layer.
- **Metadata-only instructions** (`CMD`, `ENTRYPOINT`, `ENV`, `WORKDIR`, `USER`, `EXPOSE`, `LABEL`, `HEALTHCHECK`, `ARG`) — change the image's JSON config, no new layer of bytes.

Knowing which kind each instruction is helps you reason about caching and image size.

## `FROM` — the base layer

Every Dockerfile starts with `FROM`. It picks the base image that everything else is layered on top of.

```dockerfile
FROM python:3.12-slim
```

A few rules of thumb for picking a base:

- **Official images** (no namespace, just `python`, `node`, `golang`) are maintained by Docker or the language teams and are the default choice.
- **`-slim` variants** drop everything not strictly needed for the language runtime. Smaller, faster pulls, fewer CVEs to patch.
- **`-alpine` variants** use Alpine Linux (musl libc, busybox) instead of Debian. Smallest of all, but you may hit compatibility issues with binary Python wheels (musl vs glibc) — test before committing.
- **`scratch`** is the literal empty image — zero bytes, no shell, no libc. Useful as a base for statically-linked Go or Rust binaries. The smallest possible image.
- **Pin the tag.** `FROM python:3.12-slim` is fine for dev; `FROM python:3.12.7-slim-bookworm` is better for repeatable builds. Some teams pin all the way to a digest: `FROM python@sha256:abc123...`.

A Dockerfile can also have **multiple `FROM` lines** — that's a multi-stage build, covered later in this notebook.

## `RUN` — execute at build time

`RUN <command>` runs a command inside a temporary container based on the current layer, then commits the result as a new layer. This is how you install packages, compile code, generate files at build time.

```dockerfile
RUN apt-get update && apt-get install -y --no-install-recommends curl ca-certificates \
    && rm -rf /var/lib/apt/lists/*
```

### Shell form vs exec form

Two syntaxes:

- **Shell form** — `RUN apt-get update && apt-get install -y curl`. The command runs through `/bin/sh -c`, so you get shell features (`&&`, pipes, redirection, variable expansion).
- **Exec form** — `RUN ["apt-get", "install", "-y", "curl"]`. The command is `exec`-ed directly; no shell. No `&&`, no `$VAR` expansion. Better for clarity in some cases; required when there's no shell in the image (e.g. `scratch` or distroless).

Shell form is the default and what you'll see 95% of the time in `RUN`.

### Why you'll see `&&` everywhere

Each `RUN` makes a new layer. Two separate `RUN apt-get update` and `RUN apt-get install` lines produce two layers — and worse, the cache from the first can become stale and the second installs from a stale package index. The canonical pattern is:

```dockerfile
RUN apt-get update \
 && apt-get install -y --no-install-recommends curl ca-certificates \
 && rm -rf /var/lib/apt/lists/*
```

One layer, fresh index, cleaned-up cache. Same trick applies to `pip install --no-cache-dir`, `npm ci && npm cache clean --force`, etc.

## `COPY` vs `ADD`

Both copy files from the **build context** (more on that shortly) into the image. They differ in scope.

**`COPY <src> <dst>`** — copy files or directories. That's it. No magic. Predictable.

```dockerfile
COPY requirements.txt /app/
COPY . /app/
```

**`ADD <src> <dst>`** — like `COPY`, plus two extra behaviours:

1. If `<src>` is a URL, `ADD` downloads it.
2. If `<src>` is a local `.tar`, `.tar.gz`, `.tar.bz2`, etc., `ADD` auto-extracts it into `<dst>`.

That sounds useful. It is also a footgun. The auto-extraction is surprising; the URL form bypasses your build cache discipline and silently re-downloads on changes. Modern best practice:

> **Use `COPY` by default. Use `ADD` only for the auto-extract case, and use `RUN curl` (or `wget`) instead of `ADD <URL>`.**

Both commands honour `--chown=user:group` to set ownership on the destination, and `--from=<stage>` to copy from another build stage (multi-stage, coming up).

## `WORKDIR`, `ENV`, `ARG`, `EXPOSE`, `USER`, `LABEL`, `HEALTHCHECK`

The supporting cast. Each one is small; together they shape how the image behaves when run.

**`WORKDIR /path`** — sets the working directory for subsequent `RUN`, `CMD`, `ENTRYPOINT`, `COPY`, and `ADD`. Equivalent to `cd /path` between instructions. Creates the directory if it doesn't exist. Prefer this over `RUN cd /path && ...` — it actually persists across instructions.

**`ENV KEY=value`** — set an environment variable that's visible at build time *and* at run time. Pinned into the image's config blob.

```dockerfile
ENV PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1
```

**`ARG NAME[=default]`** — build-time-only variable, *not* present at run time. Passed in with `docker build --build-arg NAME=value`. Useful for things like the version of a dependency, the target architecture, or a build-time feature flag.

```dockerfile
ARG APP_VERSION=dev
RUN echo "building version: $APP_VERSION"
```

**`EXPOSE 8080`** — *documentation* that the container listens on port 8080. **It does not publish the port.** Publishing happens at run time with `-p` (notebook 05). `EXPOSE` is metadata — `docker run -P` (capital P) uses it to auto-publish.

**`USER appuser`** — set the default user (and optionally group) that subsequent `RUN` instructions and the container's main process run as. Default is `root`, which you don't want in production. Usually paired with a `RUN useradd appuser` earlier.

**`LABEL key=value`** — attach arbitrary metadata to the image. Conventional labels (`org.opencontainers.image.source`, `.version`, `.licenses`) are read by registries and tools.

**`HEALTHCHECK [options] CMD <command>`** — tell Docker how to probe whether the container is healthy. Daemon runs the command at a configurable interval; the container's status reflects pass/fail.

```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
  CMD curl -fsS http://localhost:8080/health || exit 1
```

When health-checked, `docker ps` shows `(healthy)` or `(unhealthy)` next to the status. Orchestrators like Swarm and Kubernetes use this to decide when to route traffic.

## `CMD` vs `ENTRYPOINT` — the confusing pair

The two instructions everyone gets wrong at first. Both set what the container runs by default. The difference is in how they combine, and how arguments at run time override them.

**`CMD`** — sets the **default arguments**. Easily overridden by anything you pass to `docker run`.

**`ENTRYPOINT`** — sets the **fixed executable**. Arguments at `docker run` are appended to it.

The mental model:

```
final command run by container =  ENTRYPOINT + (run-time args OR CMD)
```

If you pass arguments to `docker run image arg1 arg2`, they replace `CMD` but are appended to `ENTRYPOINT`.

### The four combinations

| Dockerfile | `docker run img` runs | `docker run img foo bar` runs |
|---|---|---|
| `CMD ["python", "app.py"]` | `python app.py` | `foo bar` |
| `ENTRYPOINT ["python", "app.py"]` | `python app.py` | `python app.py foo bar` |
| `ENTRYPOINT ["python"]` + `CMD ["app.py"]` | `python app.py` | `python foo bar` |
| `CMD echo hi` (shell form) | `/bin/sh -c "echo hi"` | `foo bar` (shell-form CMD is dropped wholesale) |

### Practical rules

- **For "runs a single program, takes flags from the user"** (think `curl`, `git`, `aws`) — use `ENTRYPOINT` for the program, `CMD` for default arguments.
- **For "runs a long-lived service that takes its config from env vars"** (think a web app, a daemon) — just use `CMD ["python", "app.py"]` and don't set `ENTRYPOINT`. The user can override with `docker run image bash` to drop into a shell.
- **Always prefer exec form** (`["python", "app.py"]`). Shell form forks `/bin/sh -c`, which means your process is PID 2 under a shell at PID 1, and signals (like `SIGTERM` from `docker stop`) don't reach your app cleanly.

### The `--init` flag

A related gotcha: even with exec form, your PID 1 process is solely responsible for reaping zombie children and handling signals. Most app runtimes (Python, Node) aren't designed to be PID 1. Run with `docker run --init` (or set `init: true` in Compose) and Docker injects `tini` as PID 1 — it does the right thing with signals and zombies and `exec`s your process.

## The build context

When you run `docker build -t myimage .`, that final `.` is the **build context** — the directory whose contents the daemon can see during the build.

```bash
docker build -t myimage .
              path     ^
                       \__ this is the build context
```

Here's what most people miss: **the entire build context is tarballed and sent to the daemon** before the build starts. This matters because:

- If your build context is `~/projects` (which contains 12 GB of node_modules across all your repos), you'll wait minutes for the upload even if your Dockerfile only `COPY`s one file.
- `COPY` and `ADD` can only see files **inside the context**. `COPY ../shared/lib /app/lib` does *not* work — anything above the context root is invisible.

Two implications:

1. **Set the context as narrowly as possible.** Run `docker build` from the project root, not from `~`.
2. **Exclude everything the build doesn't need** with `.dockerignore`.

## `.dockerignore`

A `.dockerignore` file at the root of the build context tells the daemon which files to leave out of the tarball it ships. Syntax is `.gitignore`-style — one pattern per line, `!` for negation, `#` for comments.

```
# .dockerignore
.git
.gitignore
node_modules
__pycache__
*.pyc
.env
.env.*
.venv
dist
build
*.md
!README.md
.DS_Store
.vscode
.idea
```

Three reasons this matters:

1. **Faster builds.** Smaller context = less to upload to the daemon.
2. **Better cache hits.** A `COPY . /app/` instruction's layer is invalidated by *any* change in the context. Excluding noise (editor files, local `.env`s, build artefacts) means the cache survives unrelated changes.
3. **Security.** Stops you from accidentally baking secrets, SSH keys, or `.git/` history into your image. The number of public Docker Hub images leaking `.env` files is genuinely embarrassing — `.dockerignore` is your first defence.

Treat `.dockerignore` as part of every Dockerfile, not optional.

## A real build — Python Flask app

Let's put it together. We'll build and run a tiny Flask app. Three files: `app.py`, `requirements.txt`, `Dockerfile`. Plus a `.dockerignore`.

```python
# app.py
from flask import Flask
app = Flask(__name__)

@app.get("/")
def root():
    return {"message": "hello from docker"}

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)
```

```
# requirements.txt
flask==3.0.3
```

```dockerfile
# Dockerfile
FROM python:3.12-slim

# Don't write .pyc files; flush stdout immediately
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1

WORKDIR /app

# Install deps first -- this layer caches across source-code changes
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy source -- this layer invalidates when code changes
COPY app.py .

# Create and switch to a non-root user
RUN useradd --create-home --shell /bin/bash appuser
USER appuser

EXPOSE 8080

CMD ["python", "app.py"]
```

```
# .dockerignore
.git
__pycache__
*.pyc
.venv
.env
```

The cells below create the files, build, and run.

In [ ]:
import os, textwrap, pathlib
demo = pathlib.Path('/tmp/docker-flask-demo')
demo.mkdir(exist_ok=True)
(demo / 'app.py').write_text(textwrap.dedent('''
    from flask import Flask
    app = Flask(__name__)

    @app.get("/")
    def root():
        return {"message": "hello from docker"}

    if __name__ == "__main__":
        app.run(host="0.0.0.0", port=8080)
''').lstrip())
(demo / 'requirements.txt').write_text('flask==3.0.3\n')
(demo / 'Dockerfile').write_text(textwrap.dedent('''
    FROM python:3.12-slim
    ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
    WORKDIR /app
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt
    COPY app.py .
    RUN useradd --create-home --shell /bin/bash appuser
    USER appuser
    EXPOSE 8080
    CMD ["python", "app.py"]
''').lstrip())
(demo / '.dockerignore').write_text('.git\n__pycache__\n*.pyc\n.venv\n.env\n')
print('files written to', demo)
print(sorted(p.name for p in demo.iterdir()))

In [ ]:
# Build the image
!cd /tmp/docker-flask-demo && docker build -t flask-demo:0.1 .
!echo '---'
!docker images flask-demo

In [ ]:
# Run it and curl it
!docker run -d --name flask-demo --rm -p 8089:8080 flask-demo:0.1
!sleep 1
!curl -s http://localhost:8089/
!echo
!echo '---'
!docker stop flask-demo

### Reading the build output

When you ran `docker build`, you saw something like:

```
 => [internal] load build definition from Dockerfile
 => [internal] load metadata for docker.io/library/python:3.12-slim
 => [1/6] FROM docker.io/library/python:3.12-slim@sha256:abc...
 => [internal] load build context
 => [2/6] WORKDIR /app
 => [3/6] COPY requirements.txt .
 => [4/6] RUN pip install --no-cache-dir -r requirements.txt
 => [5/6] COPY app.py .
 => [6/6] RUN useradd --create-home --shell /bin/bash appuser
 => exporting to image
 => => writing image sha256:def456...
 => => naming to docker.io/library/flask-demo:0.1
```

Each `[n/N]` line is an instruction -> a layer. If you re-build *without changing anything*, you'll see `CACHED` next to each step and the whole thing finishes in milliseconds. If you change `app.py`, only steps 5 and 6 re-execute — earlier layers are still cache-hits. That's the layer cache in action.

The big rule that follows: **put instructions that change rarely (installing dependencies) *before* instructions that change often (copying source).** That's why `COPY requirements.txt` and `pip install` come *before* `COPY app.py`.

## Multi-stage builds

Compiled languages (Go, Rust, Java, C++) need a toolchain at build time but not at run time. You don't want your production image to ship `gcc`, build headers, or the JDK. Multi-stage builds solve this.

The idea: write multiple `FROM` blocks in one Dockerfile. The final image contains only the last stage; earlier stages are scratch pads.

```dockerfile
# ----- builder stage -----
FROM golang:1.23 AS builder
WORKDIR /src
COPY go.mod go.sum ./
RUN go mod download
COPY . .
RUN CGO_ENABLED=0 GOOS=linux go build -o /out/server ./cmd/server

# ----- runtime stage -----
FROM gcr.io/distroless/static:nonroot
COPY --from=builder /out/server /server
USER nonroot:nonroot
EXPOSE 8080
ENTRYPOINT ["/server"]
```

What's happening:

- **Stage 1 (`builder`)** uses a fat `golang:1.23` image — has the compiler, source code, dependency cache. Builds the binary into `/out/server`.
- **Stage 2** starts fresh from `distroless/static:nonroot` (a few MB, no shell, no package manager, non-root user). `COPY --from=builder` pulls in just the compiled binary. Final image ships only the binary.

A Go service that's 1.5 GB with the SDK becomes ~15 MB. Faster pulls, fewer CVEs in the image, no toolchain to attack.

The pattern applies anywhere there's a "build" phase distinct from a "run" phase:

- **Node** — install all `npm` deps + build in stage 1, copy `dist/` + production `node_modules` to a slim stage 2.
- **Python** — `pip install` into a venv in stage 1, copy the venv into a slim stage 2.
- **Java** — `mvn package` in stage 1, copy the `.jar` into `eclipse-temurin:21-jre` (no JDK) in stage 2.

You can also `--target=builder` on the `docker build` command to build only up to a named stage, which is handy for CI separation (test in builder, deploy from runtime).

## BuildKit and `docker buildx`

The original Docker build engine was straightforward but limited — strictly sequential, no parallelism, no cross-platform builds, awkward secret handling. Since 2019, Docker has shipped a new build backend called **BuildKit**, and on modern installs (Docker 23+ on Linux, Docker Desktop everywhere) it's the default.

What BuildKit changes:

- **Parallel build steps.** Multi-stage builds can build independent stages in parallel.
- **Better caching.** Smarter dependency tracking; can cache mounts (`RUN --mount=type=cache,target=/root/.cache/pip pip install ...` reuses pip's cache across builds without baking it into a layer).
- **Secrets.** `RUN --mount=type=secret,id=mysecret cat /run/secrets/mysecret` lets a build read a secret without it being committed to a layer. Pair with `docker build --secret id=mysecret,src=./secret.txt`.
- **SSH forwarding.** `RUN --mount=type=ssh git clone ...` uses your host's SSH agent for private repos.
- **Cross-platform builds.** Build `linux/amd64` and `linux/arm64` images from one Dockerfile, on one machine.

The CLI that drives BuildKit is **`docker buildx`** — an extension to `docker build` that's now the default builder. Most of the time `docker build` and `docker buildx build` are interchangeable.

A multi-platform build, end to end:

```bash
# Create a builder that supports multiple platforms (one-time setup)
docker buildx create --name multi --use --bootstrap

# Build for two architectures and push to a registry
docker buildx build \
  --platform linux/amd64,linux/arm64 \
  -t myorg/myapp:1.0.0 \
  --push \
  .
```

The `--push` flag is required for multi-platform builds because the resulting *manifest list* (one image reference, multiple per-arch images underneath) can only live in a registry, not in your local image store.

To enable advanced Dockerfile features (`RUN --mount`, heredocs, `# syntax=` directives), include a syntax pragma as the first line:

```dockerfile
# syntax=docker/dockerfile:1.7
FROM ...
```

That pulls the latest Dockerfile frontend at build time.

## Tagging conventions

Tagging is policy, not technology — Docker doesn't care what you write after the colon. But teams that ship images converge on a small set of patterns.

### What `:latest` actually means

Nothing. It's the default tag when you don't specify one. **`latest` is not "the newest version" — it's a tag named "latest"** that someone chose to push (or not). Treat it as a convenience for `hello-world` and dev-only images. **Do not deploy `:latest` to production** — your prod deploy will be "whatever happened to be there when the pod restarted," which is the opposite of reproducible.

### Tagging patterns you'll see in the wild

- **Semantic version.** `myapp:1.4.2` — clean, what humans want to read.
- **Sliding semver.** `myapp:1.4`, `myapp:1`, `myapp:latest` all repoint to the newest patch / minor / overall. Common for libraries (`python:3.12`, `node:20`). Consumers choose how aggressively they pin.
- **Git SHA.** `myapp:a1b2c3d` — uniquely identifies the source commit that built it. Great for CI; not great for humans.
- **Date.** `myapp:2026-06-03` — when versions don't apply (e.g. snapshot of a nightly build).
- **Combined.** `myapp:1.4.2-a1b2c3d` — both, for the best of both.
- **Channel.** `myapp:stable`, `myapp:edge`, `myapp:rc` — repointed as new builds promote through CI.

### Pin by digest in production

For the strictest reproducibility, deploy by digest:

```yaml
image: myorg/myapp@sha256:abc123...
```

A tag can be re-pushed; a digest cannot. Kubernetes manifests committed to git that pin by digest give you *byte-identical* rollbacks years later.

### Re-tagging an existing image

`docker tag` is free — it just adds another name pointing at the same digest:

```bash
docker tag flask-demo:0.1 myorg/flask-demo:0.1
docker tag flask-demo:0.1 myorg/flask-demo:latest
docker push myorg/flask-demo:0.1
docker push myorg/flask-demo:latest
```

Both pushes upload zero new bytes for the layers — they're content-addressed and already on the registry — they just update the tag pointers.

In [ ]:
# Demonstrate tagging: same digest, two names
!docker tag flask-demo:0.1 flask-demo:demo-alias
!docker images flask-demo
!echo '---'
# The image IDs are identical -- it's the same image with two tags
!docker image inspect flask-demo:0.1 flask-demo:demo-alias --format '{{.Id}}'